# 13·14차시 시험 대비: 모든 줄 주석 + 변형 + 예상 문제
13차시(신경망 기초)와 14차시(CNN 옷 사진) 노트북 내용을 **한 줄도 빠짐없이 주석**을 달아 정리했습니다.
각 원본 코드 아래에 **🔁 변형** 셀이 붙어 있습니다. 시험에서 조건이 바뀌면 비슷한 변형을 찾아 숫자만 바꿔 쓰세요.

**위에서부터 순서대로 실행하세요.** (변형 셀은 원본 변수 `model`, `history`를 덮어쓰지 않도록 이름을 다르게 지었습니다.)

### 목차
- **A. 공식 정리** (출력 크기, 파라미터 계산) ← 개념 문제용
- **1. 사진은 숫자표다** (cv2, shape, 슬라이싱, resize)
- **2. 필터 체험** (filter2D, 내 필터)
- **3. 데이터 준비** (mnist / fashion_mnist, 정규화, 채널 reshape)
- **4. Dense 모델** (13차시: Flatten + Dense)
- **5. CNN 모델** (14차시: Conv2D + MaxPooling2D)
- **6. 학습시키기** (compile, fit, epochs)
- **7. 학습 곡선** (loss / accuracy)
- **8. 두 모델 비교** (val_accuracy, count_params)
- **9. AI의 답 확인** (predict, argmax, 틀린 사진, 내 사진)
- **10. 개념 이해 예상 문제 + 답**
- **11. 논술형 예상 문제 + 모범 답안**

**Ctrl + F로 찾기 좋은 단어**: `파라미터 계산`, `출력 크기`, `노드`, `은닉층`, `필터 수`, `5x5`, `2층`, `풀링 없이`, `컬러`, `채널 3`, `epoch`, `과대적합`, `틀린 사진`, `내 사진`, `논술`

> ⚠ 표시가 붙은 변형은 **13·14차시에 직접 나오지 않은** 대비용입니다 (예: 컬러 사진). 문제에서 요구할 때만 쓰세요.

# A. 공식 정리 (개념 문제·파라미터 계산 문제용)

### A-1. 출력 크기 (모양이 어떻게 바뀌나)
| 층 | 들어가는 모양 | 나오는 모양 | 규칙 |
|---|---|---|---|
| `Input(shape=(28, 28, 1))` | - | `(28, 28, 1)` | (높이, 너비, 채널). 흑백 채널 1, 컬러 채널 3 |
| `Conv2D(32, (3, 3))` | `(28, 28, 1)` | `(26, 26, 32)` | 가로·세로 = **28 - 3 + 1 = 26**, 마지막 = **필터 수** |
| `Conv2D(32, (5, 5))` | `(28, 28, 1)` | `(24, 24, 32)` | 28 - 5 + 1 = 24 |
| `MaxPooling2D((2, 2))` | `(26, 26, 32)` | `(13, 13, 32)` | 가로·세로 **절반** (홀수면 버림: 11 → 5), 채널 그대로 |
| `Flatten()` | `(13, 13, 32)` | `(5408,)` | **모두 곱하기**: 13 × 13 × 32 = 5408 |
| `Dense(10)` | `(5408,)` | `(10,)` | 노드 수 |

### A-2. 파라미터 계산
| 층 | 공식 | 예시 |
|---|---|---|
| `Dense` | **(들어오는 수 + 1) × 노드 수** | Flatten 784 → Dense 128: (784 + 1) × 128 = **100,480** |
| `Conv2D` | **(필터 높이 × 필터 너비 × 들어오는 채널 + 1) × 필터 수** | 3×3, 채널 1, 필터 32: (3×3×1 + 1) × 32 = **320** |
| `Flatten`, `MaxPooling2D`, `Input` | 학습할 값 없음 | **0** |

`+ 1`은 노드(필터)마다 하나씩 더하는 값(편향, bias)입니다.

### A-3. 수업에 나온 모델들 파라미터 (직접 계산해서 확인한 값)
| 모델 | 층별 파라미터 | 합계 |
|---|---|---|
| 13차시 Dense 128 | Dense(128) 100,480 + Dense(10) 1,290 | **101,770** |
| 13차시 Dense 16 | Dense(16) 12,560 + Dense(10) 170 | **12,730** |
| 14차시 CNN | Conv2D 320 + Dense(10) 54,090 | **54,410** |

### A-4. 변형 모델 파라미터 (미리 계산해 둔 값)
| 변형 | 출력 모양 흐름 | 합계 |
|---|---|---|
| Dense 64개 | 784 → 64 → 10 | 50,890 |
| Dense 256개 | 784 → 256 → 10 | 203,530 |
| Dense 2층 (128 → 64) | 784 → 128 → 64 → 10 | 109,386 |
| CNN 필터 16개 | (26,26,16) → (13,13,16) → 2704 → 10 | 27,210 |
| CNN 필터 64개 | (26,26,64) → (13,13,64) → 10816 → 10 | 108,810 |
| CNN 필터 5x5 | (24,24,32) → (12,12,32) → 4608 → 10 | 46,922 |
| CNN 2층 (32 → 64) | (26,26,32) → (13,13,32) → (11,11,64) → (5,5,64) → 1600 → 10 | 34,826 |
| CNN + Dense(64) 은닉층 | ... → 5408 → 64 → 10 | 347,146 |
| CNN 풀링 없이 | (26,26,32) → 21632 → 10 | 216,650 |
| ⚠ 컬러 28x28x3 CNN | Conv2D 896 (= (3×3×3+1)×32) | 54,986 |
| ⚠ 컬러 32x32x3 CNN | (30,30,32) → (15,15,32) → 7200 → 10 | 72,906 |

# 1. 사진은 숫자표다 (13차시)
컴퓨터에게 사진은 **0~255 숫자가 들어 있는 표(배열)** 입니다. 0은 검은색, 255는 하얀색.

In [ ]:
# 라이브러리 불러오고 예제 사진 파일 만들기
import os                                   # 파일이 있는지 확인하는 도구
import cv2                                  # 사진을 읽고, 크기를 바꾸고, 필터를 적용하는 도구 (OpenCV)
import numpy as np                          # 숫자표(배열)를 만들고 계산하는 도구
import matplotlib.pyplot as plt             # 사진과 그래프를 그리는 도구
from skimage import data                    # 예제 사진(고양이)이 들어 있는 도구
if not os.path.exists('sample.png'):        # sample.png 파일이 아직 없으면 (있으면 덮어쓰지 않음)
    cv2.imwrite('sample.png', cv2.cvtColor(data.chelsea(), cv2.COLOR_RGB2BGR))   # 고양이 사진을 파일로 저장 (cv2는 BGR 순서라서 RGB -> BGR로 바꿈)

In [ ]:
# 사진 파일을 흑백으로 불러와서 보기
filename = 'sample.png'                                 # 불러올 사진 파일 이름 (내 사진이면 'me.jpg' 처럼 바꾸기)
img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)        # 사진을 흑백으로 읽어서 숫자표로 가져옴
print(img is None)                                      # False 이면 정상, True 이면 파일 이름이 틀린 것
plt.imshow(img, cmap='gray')                            # 숫자표를 흑백 사진으로 그리기
plt.axis('off')                                         # 눈금(축) 숨기기
plt.show()                                              # 화면에 보여주기

In [ ]:
# 사진의 크기와 왼쪽 위 5x5 칸의 숫자 보기
print(img.shape)                # (높이, 너비) -> (300, 451) : 높이 300칸, 너비 451칸
print(img[:5, :5])              # [위에서 5줄, 왼쪽에서 5칸] 숫자 보기 (0~255 사이)

#### 🔁 변형 ①: 다른 위치의 숫자 보기 / 가장 밝은 값·어두운 값

In [ ]:
print(img[100:105, 200:205])    # 위에서 100~104줄, 왼쪽에서 200~204칸 (슬라이싱은 끝 숫자 미포함)
print(img[0, 0])                # 맨 왼쪽 위 한 칸(픽셀)의 값
print(img.max(), img.min())     # 가장 밝은 값, 가장 어두운 값

In [ ]:
# 사진을 28x28로 줄이기
small = cv2.resize(img, (28, 28))       # cv2.resize(사진, (너비, 높이)) -> 28x28 로 줄임
print(small.shape)                      # (28, 28) -> 28 x 28 = 784개 숫자짜리 표
plt.imshow(small, cmap='gray')          # 줄인 사진 그리기 (모자이크처럼 보임)
plt.show()                              # 화면에 보여주기

#### 🔁 변형 ②: 가로·세로가 다른 크기로 줄이기 (순서 함정!)
`cv2.resize`는 **(너비, 높이)**, `shape`는 **(높이, 너비)** 로 순서가 반대입니다.

In [ ]:
wide = cv2.resize(img, (40, 20))        # 너비 40, 높이 20 으로 줄이기
print(wide.shape)                       # (20, 40) -> shape 는 (높이, 너비) 순서라 거꾸로 나옴

#### 🔁 변형 ③ ⚠: 컬러로 불러오기 (채널 3)

In [ ]:
color = cv2.imread(filename)                            # IMREAD_GRAYSCALE 을 빼면 컬러로 읽음
print(color.shape)                                      # (300, 451, 3) -> (높이, 너비, 채널) 컬러는 채널 3
color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)          # cv2는 BGR 순서라서 화면에 맞게 RGB 로 바꿈
plt.imshow(color)                                       # 컬러 사진은 cmap 없이 그림
plt.axis('off')                                         # 눈금 숨기기
plt.show()                                              # 화면에 보여주기

# 2. 필터 체험 (14차시)
**필터** = 3×3 숫자표. 사진 위를 훑으며 계산해서 선·모서리 같은 모양을 찾습니다.
`Conv2D`는 이 필터의 숫자를 **사람이 정하지 않고 학습으로 스스로 찾습니다** (필터 숫자 = 학습되는 가중치).

In [ ]:
# 필터 3개 만들기
filter_v = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])      # 세로 경계를 찾는 필터 (왼쪽 음수, 오른쪽 양수)
filter_h = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])      # 가로 경계를 찾는 필터 (위 음수, 아래 양수)
filter_b = np.ones((10, 10)) / 100                             # 흐리게(블러) 만드는 필터 (10x10 칸의 평균)

# 필터를 적용해서 비교
titles = ['original', 'vertical edge', 'horizontal edge', 'blur']                           # 사진마다 붙일 제목
images = [img] + [cv2.filter2D(img, -1, f) for f in [filter_v, filter_h, filter_b]]          # 원본 + 필터 3개를 적용한 사진 (filter2D(사진, -1, 필터))
for i in range(4):                                  # 사진 4장을 하나씩
    plt.subplot(1, 4, i + 1)                        # 1행 4열 중 i+1 번째 칸에 그리기
    plt.imshow(images[i], cmap='gray')             # 흑백으로 그리기
    plt.title(titles[i], fontsize=9)               # 제목 붙이기
    plt.axis('off')                                 # 눈금 숨기기
plt.show()                                          # 화면에 보여주기

#### 🔁 변형 ④: 내 필터 만들기 (대각선 / 블러 / 선명하게)

In [ ]:
my_filter = np.array([[ 0,  1,  2],             # 대각선(/ 방향) 경계를 찾는 필터
                      [-1,  0,  1],              # 가운데 줄
                      [-2, -1,  0]])             # 아래 줄 (왼쪽 아래 음수, 오른쪽 위 양수 -> / 방향 경계)
blur3 = np.ones((3, 3)) / 9                      # 숫자를 모두 1로 하고 9로 나누면 흐리게 만드는 필터
sharp = np.array([[-1, -1, -1],                  # 가운데 5, 나머지 -1 이면 선명하게 만드는 필터
                  [-1,  5, -1],                  # 가운데 칸을 강하게
                  [-1, -1, -1]])                 # 주변 칸은 빼기

titles = ['original', 'my filter', 'blur 3x3', 'sharp']                                     # 제목
images = [img] + [cv2.filter2D(img, -1, f) for f in [my_filter, blur3, sharp]]               # 필터 적용
for i in range(4):                                  # 4장을 하나씩
    plt.subplot(1, 4, i + 1)                        # 1행 4열
    plt.imshow(images[i], cmap='gray')             # 흑백으로 그리기
    plt.title(titles[i], fontsize=9)               # 제목
    plt.axis('off')                                 # 눈금 숨기기
plt.show()                                          # 보여주기

# 3. 데이터 준비
13차시는 손글씨 숫자 `mnist`, 14차시는 옷 사진 `fashion_mnist`. **불러오는 방법은 똑같습니다.**

In [ ]:
# 라이브러리 불러오기
from tensorflow.keras.datasets import mnist, fashion_mnist                      # 손글씨 숫자 / 옷 사진 데이터
from tensorflow.keras.models import Sequential                                  # 층을 순서대로 쌓아 모델을 만드는 틀
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense # 모델에 쌓을 층들

In [ ]:
# 옷 사진 데이터 불러오기 (학습용 60000장, 테스트용 10000장)
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()    # (학습용 사진, 학습용 정답), (테스트용 사진, 테스트용 정답)
print(X_train.shape, X_test.shape)                                  # (60000, 28, 28) (10000, 28, 28) -> (장수, 높이, 너비)
print(y_train[:10])                                                 # 앞에서 10개 정답 번호 (0~9)
print(X_train.max(), X_train.min())                                 # 255 0 -> 픽셀값 범위

# 숫자 0~9에 해당하는 옷 이름
names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',     # 0~4번 옷 이름
         'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']          # names[정답번호] 로 옷 이름을 꺼냄

In [ ]:
# 옷 사진 20장 보기
for i in range(20):                                     # 0번부터 19번 사진까지
    plt.subplot(4, 5, i + 1)                            # 4행 5열 중 i+1 번째 칸
    plt.imshow(X_train[i], cmap='gray')                 # i번째 사진을 흑백으로 그리기
    plt.title(names[y_train[i]], fontsize=8)            # 정답 번호를 옷 이름으로 바꿔서 제목으로
    plt.axis('off')                                     # 눈금 숨기기
plt.show()                                              # 보여주기

In [ ]:
# 픽셀값을 0~1로 만들기 (정규화)
X_train = X_train / 255         # 가장 큰 값 255로 나누면 0~255 -> 0~1 (가장 밝은 칸 255 -> 1.0)
X_test = X_test / 255           # 테스트용도 똑같이 나눠야 함 (한쪽만 나누면 안 됨!)
print(X_train.max())            # 1.0

In [ ]:
# 이미지에 채널 칸 붙이기: (장수, 28, 28) -> (장수, 28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)    # -1 : 장수(60000)는 알아서 채워 줌, 마지막 1 = 흑백 채널
X_test = X_test.reshape(-1, 28, 28, 1)      # 테스트용도 똑같이
print(X_train.shape)                        # (60000, 28, 28, 1)

#### 🔁 변형 ⑤: 손글씨 숫자(mnist)로 바꾸기
문제에서 **숫자**를 맞히라고 하면 데이터만 바꾸면 됩니다. (이 셀은 따로 이름을 붙여서 위 데이터를 덮어쓰지 않음)

In [ ]:
(Xd_train, yd_train), (Xd_test, yd_test) = mnist.load_data()    # 손글씨 숫자 데이터 불러오기
Xd_train = Xd_train / 255                                       # 0~1 로 만들기
Xd_test = Xd_test / 255                                         # 테스트용도
Xd_train = Xd_train.reshape(-1, 28, 28, 1)                      # CNN 에 넣으려면 채널 칸 붙이기
Xd_test = Xd_test.reshape(-1, 28, 28, 1)                        # 테스트용도
print(Xd_train.shape, yd_train[:10])                            # (60000, 28, 28, 1) [5 0 4 1 9 2 1 3 1 4]
# 숫자 데이터는 정답 번호가 곧 숫자라서 names 리스트가 필요 없음

#### 🔁 변형 ⑥ ⚠: 컬러 사진이면? (채널 3)
컬러는 `(높이, 너비, 3)`이라 **reshape 할 필요 없이** 이미 채널 칸이 있습니다. `Input(shape=(높이, 너비, 3))`로만 바꾸면 됩니다.
```python
X_train = X_train / 255                       # 정규화는 똑같이 255로 나누기
print(X_train.shape)                          # 예) (50000, 32, 32, 3)
# 만약 채널 칸이 없다면: X_train = X_train.reshape(-1, 32, 32, 3)
```

# 4. Dense 모델 (13차시: 한 줄로 펼쳐서 맞히기)
`입력(28x28) → Flatten(784) → Dense(128, relu) → Dense(10, softmax)`
| 용어 | 뜻 |
|---|---|
| 층(layer) | 신경망을 이루는 한 단계. 층을 **쌓아서** 모델을 만든다 |
| 노드 | 층 안의 계산 단위. `Dense(128)` = 노드 128개 |
| 은닉층 | 입력층과 출력층 사이의 층 |
| `relu` | 음수는 0, 양수는 그대로 → 은닉층에 사용 |
| `softmax` | 출력값을 **합이 1인 확률**로 바꿈 → 출력층에 사용 |
| 출력층 노드 수 | **분류할 종류의 수** (0~9 → 10개) |

In [ ]:
# 13차시 모델(Flatten + Dense) 만들기
dense_model = Sequential()                                  # 층을 쌓을 빈 모델 틀
dense_model.add(Input(shape=(28, 28, 1)))                   # 입력 모양 (높이 28, 너비 28, 채널 1)
dense_model.add(Flatten())                                  # 28x28x1 이미지를 한 줄(784개)로 펼치기
dense_model.add(Dense(128, activation='relu'))              # 노드 128개짜리 은닉층
dense_model.add(Dense(10, activation='softmax'))            # 출력층: 10종류 각각일 확률
dense_model.summary()                                       # 층마다 출력 모양과 파라미터 수 -> 합계 101,770

#### 🔁 변형 ⑦: 노드 수 바꾸기 (16 / 64 / 256)
노드 수만 바꾸면 됩니다. 파라미터: 16개 → 12,730 / 64개 → 50,890 / 256개 → 203,530

In [ ]:
small_model = Sequential()                                  # 빈 모델 틀
small_model.add(Input(shape=(28, 28, 1)))                   # 입력 모양
small_model.add(Flatten())                                  # 한 줄로 펼치기 (784)
small_model.add(Dense(16, activation='relu'))               # 노드 16개짜리 은닉층 (128 -> 16 으로 줄임)
small_model.add(Dense(10, activation='softmax'))            # 출력층
small_model.summary()                                       # 합계 12,730

#### 🔁 변형 ⑧: 은닉층 2개 쌓기 (128 → 64)

In [ ]:
deep_model = Sequential()                                   # 빈 모델 틀
deep_model.add(Input(shape=(28, 28, 1)))                    # 입력 모양
deep_model.add(Flatten())                                   # 한 줄로 펼치기 (784)
deep_model.add(Dense(128, activation='relu'))               # 은닉층 1: 노드 128개
deep_model.add(Dense(64, activation='relu'))                # 은닉층 2: 노드 64개 (층을 하나 더 쌓음)
deep_model.add(Dense(10, activation='softmax'))             # 출력층
deep_model.summary()                                        # 합계 109,386

#### 🔁 변형 ⑨: 13차시처럼 채널 없는 (28, 28) 입력
데이터를 reshape 하지 **않았다면** `Input(shape=(28, 28))`로 씁니다. (파라미터 수는 같음: 101,770)
```python
model.add(Input(shape=(28, 28)))      # 채널 칸이 없는 (장수, 28, 28) 데이터일 때
```

# 5. CNN 모델 (14차시: 모양을 보면서 맞히기)
순서: **필터로 훑기(`Conv2D`) → 크기 줄이기(`MaxPooling2D`) → 펼치기(`Flatten`) → 맞히기(`Dense`)**

In [ ]:
# CNN 모델 만들기: 입력 → 훑기 → 줄이기 → 펼치기 → 출력
model = Sequential()                                        # 층을 쌓을 빈 모델 틀
model.add(Input(shape=(28, 28, 1)))                         # 입력 모양 (높이 28, 너비 28, 채널 1)
model.add(Conv2D(32, (3, 3), activation='relu'))            # 3x3 필터 32개로 훑기 -> (26, 26, 32)
model.add(MaxPooling2D((2, 2)))                             # 가로·세로를 절반으로 줄이기 -> (13, 13, 32)
model.add(Flatten())                                        # 한 줄로 펼치기 -> 13 x 13 x 32 = 5408
model.add(Dense(10, activation='softmax'))                  # 출력층: 10종류 각각일 확률
model.summary()                                             # 층마다 출력 모양과 파라미터 수

목표 결과:
```
conv2d (Conv2D)                 (None, 26, 26, 32)    320       <- (3x3x1 + 1) x 32
max_pooling2d (MaxPooling2D)    (None, 13, 13, 32)    0
flatten (Flatten)               (None, 5408)          0         <- 13 x 13 x 32
dense (Dense)                   (None, 10)            54,090    <- (5408 + 1) x 10
Total params: 54,410
```

#### 🔁 변형 ⑩: 필터 수 바꾸기 (16개 / 64개)
필터 수 = `Conv2D`의 첫 번째 숫자 = 출력 모양의 마지막 숫자. 16개 → 27,210 / 64개 → 108,810

In [ ]:
cnn16 = Sequential()                                        # 빈 모델 틀
cnn16.add(Input(shape=(28, 28, 1)))                         # 입력 모양
cnn16.add(Conv2D(16, (3, 3), activation='relu'))            # 3x3 필터 16개 -> (26, 26, 16)
cnn16.add(MaxPooling2D((2, 2)))                             # 절반 -> (13, 13, 16)
cnn16.add(Flatten())                                        # 13 x 13 x 16 = 2704
cnn16.add(Dense(10, activation='softmax'))                  # 출력층
cnn16.summary()                                             # 합계 27,210

#### 🔁 변형 ⑪: 필터 크기 5x5
필터가 커지면 출력이 더 작아짐: 28 - 5 + 1 = **24**. 파라미터 (5×5×1 + 1) × 32 = 832

In [ ]:
cnn5 = Sequential()                                         # 빈 모델 틀
cnn5.add(Input(shape=(28, 28, 1)))                          # 입력 모양
cnn5.add(Conv2D(32, (5, 5), activation='relu'))             # 5x5 필터 32개 -> (24, 24, 32)
cnn5.add(MaxPooling2D((2, 2)))                              # 절반 -> (12, 12, 32)
cnn5.add(Flatten())                                         # 12 x 12 x 32 = 4608
cnn5.add(Dense(10, activation='softmax'))                   # 출력층
cnn5.summary()                                              # 합계 46,922

#### 🔁 변형 ⑫: Conv2D + MaxPooling2D 2층 쌓기 (32 → 64)
두 번째 `Conv2D`는 들어오는 채널이 **32**라서 파라미터가 (3×3×**32** + 1) × 64 = 18,496

In [ ]:
cnn2 = Sequential()                                         # 빈 모델 틀
cnn2.add(Input(shape=(28, 28, 1)))                          # 입력 모양
cnn2.add(Conv2D(32, (3, 3), activation='relu'))             # 1층: 필터 32개 -> (26, 26, 32)
cnn2.add(MaxPooling2D((2, 2)))                              # 절반 -> (13, 13, 32)
cnn2.add(Conv2D(64, (3, 3), activation='relu'))             # 2층: 필터 64개 -> (11, 11, 64)  (13 - 3 + 1 = 11)
cnn2.add(MaxPooling2D((2, 2)))                              # 절반 -> (5, 5, 64)  (11 / 2 = 5.5 -> 버림 5)
cnn2.add(Flatten())                                         # 5 x 5 x 64 = 1600
cnn2.add(Dense(10, activation='softmax'))                   # 출력층
cnn2.summary()                                              # 합계 34,826 (층은 더 많은데 파라미터는 더 적음!)

#### 🔁 변형 ⑬: CNN 뒤에 Dense 은닉층 추가

In [ ]:
cnn_d = Sequential()                                        # 빈 모델 틀
cnn_d.add(Input(shape=(28, 28, 1)))                         # 입력 모양
cnn_d.add(Conv2D(32, (3, 3), activation='relu'))            # 필터 32개 -> (26, 26, 32)
cnn_d.add(MaxPooling2D((2, 2)))                             # 절반 -> (13, 13, 32)
cnn_d.add(Flatten())                                        # 5408
cnn_d.add(Dense(64, activation='relu'))                     # 은닉층: 노드 64개 (5408 + 1) x 64 = 346,176
cnn_d.add(Dense(10, activation='softmax'))                  # 출력층
cnn_d.summary()                                             # 합계 347,146

#### 🔁 변형 ⑭: 풀링 없이 (MaxPooling2D 의 역할 확인)
풀링을 빼면 Flatten 결과가 26 × 26 × 32 = 21,632로 커져서 파라미터가 216,650으로 **4배 정도** 늘어납니다.

In [ ]:
cnn_np = Sequential()                                       # 빈 모델 틀
cnn_np.add(Input(shape=(28, 28, 1)))                        # 입력 모양
cnn_np.add(Conv2D(32, (3, 3), activation='relu'))           # 필터 32개 -> (26, 26, 32)
cnn_np.add(Flatten())                                       # 풀링 없이 바로 펼치기 -> 21632
cnn_np.add(Dense(10, activation='softmax'))                 # 출력층
cnn_np.summary()                                            # 합계 216,650

#### 🔁 변형 ⑮ ⚠: 컬러 사진용 CNN (채널 3)
`Input`의 마지막 숫자만 3으로. 첫 `Conv2D` 파라미터가 (3×3×**3** + 1) × 32 = **896**으로 바뀜.

In [ ]:
cnn_color = Sequential()                                    # 빈 모델 틀
cnn_color.add(Input(shape=(32, 32, 3)))                     # 컬러 사진: (높이 32, 너비 32, 채널 3)
cnn_color.add(Conv2D(32, (3, 3), activation='relu'))        # 필터 32개 -> (30, 30, 32)
cnn_color.add(MaxPooling2D((2, 2)))                         # 절반 -> (15, 15, 32)
cnn_color.add(Flatten())                                    # 15 x 15 x 32 = 7200
cnn_color.add(Dense(10, activation='softmax'))              # 출력층 (분류할 종류 수에 맞추기)
cnn_color.summary()                                         # 합계 72,906

#### 🔁 변형 ⑯: 분류할 종류 수가 다르면?
출력층 `Dense`의 노드 수 = **분류할 종류의 수**. 예) 3종류 → `Dense(3, activation='softmax')`
정답 번호는 0부터 시작합니다 (3종류면 0, 1, 2).

# 6. 학습시키기
- `compile`: 학습 방법 정하기 (어떻게 고칠지, 무엇을 줄일지, 무엇을 기록할지)
- `fit`: 실제로 학습 / `epochs`: 학습용 데이터 **전체를 몇 번 반복**할지
- 한 번에 32장씩(batch) 학습 → 60000장 ÷ 32 = **1875번**이 1 epoch

In [ ]:
# 학습 방법 정하기
model.compile(optimizer='adam',                             # 모델의 값을 고쳐주는 방법 (adam)
              loss='sparse_categorical_crossentropy',       # 틀린 정도(loss) 계산 방법 (정답이 0~9 번호일 때)
              metrics=['accuracy'])                         # 학습하면서 정확도도 같이 기록

In [ ]:
# 학습용 데이터 전체를 3번 반복해서 학습
history = model.fit(X_train, y_train,                       # 학습용 사진, 학습용 정답
                    epochs=3,                               # 전체를 3번 반복 (문제에서 말한 숫자로 바꾸기)
                    validation_data=(X_test, y_test))       # 매 epoch 끝에 테스트용으로 점수 확인 (val_)
# history.history 안에 epoch 마다 'loss', 'accuracy', 'val_loss', 'val_accuracy' 가 기록됨

- `loss` / `accuracy`: **학습용** 데이터 점수
- `val_loss` / `val_accuracy`: **테스트용**(처음 보는) 데이터 점수

In [ ]:
# 13차시 Dense 모델도 같은 조건으로 학습 (비교용)
dense_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])     # 학습 방법 (CNN 과 똑같이)
dense_history = dense_model.fit(X_train, y_train, epochs=3, validation_data=(X_test, y_test))          # 같은 데이터, 같은 epoch 로 학습

#### 🔁 변형 ⑰: epoch 바꾸기 / 다른 모델 학습시키기
모델 이름만 바꾸면 똑같습니다. **compile → fit** 순서는 항상 같음.
```python
cnn2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])   # 학습 방법
cnn2_history = cnn2.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))         # 5번 반복
```

# 7. 학습 곡선 그리고 읽기

In [ ]:
# loss(틀린 정도) 곡선: 학습용과 테스트용
plt.plot(history.history['loss'], label='train')            # 학습용 loss (x축 = epoch 번호 0, 1, 2)
plt.plot(history.history['val_loss'], label='test')         # 테스트용 loss (이름이 val_ 로 시작)
plt.xlabel('epoch')                                         # x축 이름
plt.ylabel('loss')                                          # y축 이름
plt.legend()                                                # label 로 붙인 이름표 표시
plt.show()                                                  # 보여주기

In [ ]:
# accuracy(정확도) 곡선: 학습용과 테스트용
plt.plot(history.history['accuracy'], label='train')        # 학습용 정확도
plt.plot(history.history['val_accuracy'], label='test')     # 테스트용 정확도
plt.xlabel('epoch')                                         # x축 이름
plt.ylabel('accuracy')                                      # y축 이름
plt.legend()                                                # 이름표 표시
plt.show()                                                  # 보여주기

#### 🔁 변형 ⑱: 두 모델의 테스트 정확도 곡선 겹쳐 그리기

In [ ]:
plt.plot(history.history['val_accuracy'], label='CNN')              # CNN 테스트 정확도
plt.plot(dense_history.history['val_accuracy'], label='Dense')      # Dense 테스트 정확도
plt.xlabel('epoch')                                                 # x축 이름
plt.ylabel('val_accuracy')                                          # y축 이름
plt.legend()                                                        # 이름표 표시
plt.show()                                                          # 보여주기

**곡선 읽는 법**
- loss는 내려가고 accuracy는 올라가면 → 학습이 잘 되고 있음
- 학습용(train) 선이 테스트용(test) 선보다 좋으면 → 이미 본 문제라 더 잘 맞힘 (정상)
- train 은 계속 좋아지는데 test 는 오히려 나빠지면 → **과대적합** (학습용을 외워버림)

# 8. 두 모델 비교

In [ ]:
# 마지막 epoch 의 테스트용 정확도와 파라미터 수 비교
print('CNN:', round(history.history['val_accuracy'][-1], 4))           # [-1] = 마지막 epoch 값, round(값, 4) = 소수 넷째 자리까지
print('Dense:', round(dense_history.history['val_accuracy'][-1], 4))   # Dense 모델의 마지막 테스트 정확도
print('CNN 파라미터:', model.count_params())                            # 54410
print('Dense 파라미터:', dense_model.count_params())                    # 101770

#### 🔁 변형 ⑲: loss 비교 / 학습용 정확도 비교
`[...]` 안의 이름만 바꾸면 됩니다: `'loss'`, `'accuracy'`, `'val_loss'`, `'val_accuracy'`
```python
print('CNN 테스트 loss:', round(history.history['val_loss'][-1], 4))     # 테스트 loss (작을수록 좋음)
print('CNN 학습 정확도:', round(history.history['accuracy'][-1], 4))      # 학습용 정확도
print('CNN 첫 epoch:', round(history.history['val_accuracy'][0], 4))     # [0] = 첫 번째 epoch
```

# 9. AI의 답 확인하기
`predict` → 10종류 각각의 확률 → `argmax(axis=1)` → **가장 확률이 높은 번호**

In [ ]:
# 테스트 사진 10장: 정답과 AI의 답
pred = model.predict(X_test[:10], verbose=0).argmax(axis=1)     # 앞 10장의 확률 중 가장 큰 번호 (verbose=0 : 진행 표시 숨김)
for i in range(10):                                             # 10장을 하나씩
    plt.subplot(2, 5, i + 1)                                    # 2행 5열
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')          # 채널 칸을 빼서 (28, 28) 로 그리기
    plt.title(names[y_test[i]] + ' -> ' + names[pred[i]], fontsize=7)   # '정답 -> AI의 답'
    plt.axis('off')                                             # 눈금 숨기기
plt.show()                                                      # 보여주기
print('맞힌 개수:', (pred == y_test[:10]).sum(), '/ 10')        # 같으면 True(1) -> 더하면 맞힌 개수

In [ ]:
# AI가 틀린 사진 9장 보기
pred_all = model.predict(X_test, verbose=0).argmax(axis=1)      # 테스트 사진 10000장 전부 예측
wrong = np.where(pred_all != y_test)[0]                         # 예측과 정답이 다른 사진의 번호들
print('틀린 사진 수:', len(wrong), '/', len(y_test))             # 틀린 개수 / 전체
for k in range(min(9, len(wrong))):                             # 틀린 사진 중 앞 9장 (9장보다 적으면 그만큼)
    i = wrong[k]                                                # k번째 틀린 사진의 번호
    plt.subplot(3, 3, k + 1)                                    # 3행 3열
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')          # 그리기
    plt.title('answer ' + names[y_test[i]] + ' / AI ' + names[pred_all[i]], fontsize=7)    # 정답 / AI 의 답
    plt.axis('off')                                             # 눈금 숨기기
plt.show()                                                      # 보여주기

In [ ]:
# 내 사진(고양이)을 AI에게 보여주기
filename = 'sample.png'                                         # 내 사진이면 파일 이름 바꾸기
photo = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)              # 흑백으로 읽기
photo = cv2.resize(photo, (28, 28)) / 255                       # 28x28 로 줄이고 0~1 로 (학습 데이터와 똑같이!)
# photo = 1 - photo                                             # 배경이 흰색인 사진이면 앞의 # 지우기 (옷 데이터는 배경이 검정)
prob = model.predict(photo.reshape(1, 28, 28, 1), verbose=0)[0] # (1장, 28, 28, 1) 모양으로 넣고 첫 번째 결과(10개 확률)
print('AI의 답:', names[prob.argmax()], '| 확신:', round(prob.max() * 100), '%')    # 가장 큰 확률의 이름, 그 확률(%)
plt.imshow(photo, cmap='gray')                                  # 사진 보기
plt.axis('off')                                                 # 눈금 숨기기
plt.show()                                                      # 보여주기

#### 🔁 변형 ⑳: 특정 한 장만 예측하기
```python
i = 0                                                                   # 확인할 사진 번호
prob = model.predict(X_test[i].reshape(1, 28, 28, 1), verbose=0)[0]     # 한 장도 (1, 28, 28, 1) 모양으로 넣어야 함
print('정답:', names[y_test[i]], '| AI:', names[prob.argmax()])         # 정답과 AI 의 답
```

# 10. 개념 이해 예상 문제 + 답

**Q1. 필터란 무엇이고, `Conv2D`에서 필터는 어떻게 정해지나요?**
필터는 이미지 위를 훑으며 계산하는 **작은 숫자표(예: 3×3)** 로, 숫자 배치에 따라 세로선·가로선·대각선 같은 서로 다른 모양을 찾는다. 14차시 필터 체험에서는 사람이 숫자를 정했지만, `Conv2D`에서는 필터의 숫자를 **학습을 통해 모델이 스스로 찾는다** (필터 숫자 = 학습되는 가중치).

**Q2. `MaxPooling2D((2, 2))`의 역할은?**
2×2 칸마다 **가장 큰 값 하나만 남겨** 이미지의 가로·세로를 **절반**으로 줄인다. 예) (26, 26, 32) → (13, 13, 32). 채널 수는 그대로이고, 학습할 파라미터는 **0개**이다. 크기가 줄어서 계산량과 뒤쪽 Dense 층의 파라미터가 줄어든다.

**Q3. 채널이란? 왜 `reshape(-1, 28, 28, 1)`을 하나요?**
채널은 한 칸(픽셀)에 들어 있는 색 정보의 수로, 흑백은 **1**, 컬러(RGB)는 **3**이다. `Conv2D`는 이미지가 `(높이, 너비, 채널)` 모양이길 원하는데, `fashion_mnist`는 `(장수, 28, 28)`이라 채널 칸이 없어서 마지막에 1을 붙인다. `-1`은 장수(60000)를 알아서 채워 달라는 뜻이다.

**Q4. `Conv2D(32, (3, 3))`의 출력 모양이 (26, 26, 32)인 이유는?**
3×3 필터가 28칸 위를 미끄러질 수 있는 위치는 **28 - 3 + 1 = 26**개라서 가로·세로가 26이 되고, 필터 **32개**가 각각 결과 한 장씩을 만들어서 마지막이 32이다.

**Q5. `Conv2D` 층의 파라미터가 320개인 이유는?**
필터 하나에 3 × 3 × 1(채널) = 9개 숫자 + 더하는 값 1개 = 10개, 필터가 32개이므로 **10 × 32 = 320**.

**Q6. Flatten 층의 역할과 출력 크기는?**
여러 겹의 이미지 모양을 **한 줄로 펼쳐** `Dense` 층에 넣을 수 있게 만든다. 모든 숫자를 곱한 크기가 된다: (13, 13, 32) → 13 × 13 × 32 = **5408**. 파라미터는 0개.

**Q7. 출력층이 `Dense(10, activation='softmax')`인 이유는?**
옷(숫자) 종류가 **10가지**라서 노드가 10개이고, `softmax`는 10개의 출력값을 **합이 1인 확률**로 바꿔 준다. 가장 확률이 큰 번호(`argmax`)가 AI의 답이다.

**Q8. `relu`는 무엇인가요?**
음수는 0으로, 양수는 그대로 내보내는 활성화 함수로 은닉층(`Conv2D`, 중간 `Dense`)에 쓴다.

**Q9. epoch이란? 3 epoch 은 몇 번 계산하나요?**
학습용 데이터 **전체를 한 바퀴** 학습하는 것. 기본으로 32장씩 묶어(batch) 학습하므로 60000 ÷ 32 = 1875번이 1 epoch, 3 epoch 은 1875 × 3 = **5625번** 고친다.

**Q10. 픽셀값을 255로 나누는 이유는?**
픽셀값 0~255를 **0~1** 범위로 맞춰서(정규화) 숫자 크기를 작고 고르게 만들면 학습이 안정적이다. 학습용과 테스트용 **모두** 똑같이 나눠야 한다.

**Q11. `loss`와 `val_loss`의 차이는?**
`loss`는 **학습용** 데이터에서 틀린 정도, `val_loss`는 **테스트용**(학습에 안 쓴, 처음 보는) 데이터에서 틀린 정도. 실력은 `val_`로 판단한다.

**Q12. CNN이 Dense 모델보다 이미지를 잘 맞히는 이유는?**
Dense 모델은 사진을 처음부터 한 줄로 펼쳐서 **이웃한 픽셀끼리의 위치 관계(모양)** 를 잃지만, CNN은 필터로 사진을 훑으며 **선·모서리 같은 모양을 그대로 찾고**, 같은 필터를 사진 전체에 다시 써서 파라미터도 적다.

# 11. 논술형 예상 문제 + 모범 답안
**쓰는 순서: 결과(숫자) → 해석(왜) → 근거/결론**
⚠ 정확도 숫자는 실행할 때마다 조금씩 다릅니다. **`0.9xxx` 자리는 내 실행 결과로 바꿔 쓰세요.** (아래는 보통 나오는 값)

---
### 논술 1. CNN 모델과 Dense 모델의 결과를 비교하고 해석하시오.
**결과**: 같은 옷 사진 데이터로 3 epoch 학습했을 때 CNN의 테스트 정확도는 약 **0.90**, Dense 모델은 약 **0.87**로 CNN이 더 높았다. 파라미터 수는 CNN이 **54,410개**, Dense가 **101,770개**로 CNN이 절반 정도였다.

**해석**: Dense 모델은 사진을 처음부터 한 줄(784개)로 펼치기 때문에 이웃한 픽셀끼리의 위치 관계, 즉 옷의 모양 정보를 잃는다. 반면 CNN은 3×3 필터가 사진을 훑으며 선·모서리 같은 모양을 찾고, 같은 필터를 사진 전체에 다시 사용하므로 파라미터가 적어도 모양을 잘 파악한다. 또 MaxPooling2D로 크기를 절반으로 줄여 뒤쪽 층의 파라미터도 줄었다.

**결론**: 이미지처럼 모양이 중요한 데이터에는 파라미터 수가 많은 모델보다, 모양을 볼 수 있는 구조(CNN)가 더 효과적이다.

---
### 논술 2. 학습 곡선(loss, accuracy)을 보고 학습 과정을 해석하시오.
**결과**: epoch이 지날수록 학습용과 테스트용 loss는 모두 **내려가고**, accuracy는 모두 **올라갔다**. 마지막 epoch에서 학습용 정확도(약 **0.92**)가 테스트용 정확도(약 **0.90**)보다 조금 높았다.

**해석**: loss가 줄고 정확도가 오른 것은 모델이 학습을 반복하며 틀린 정도를 줄이는 방향으로 필터와 가중치를 고쳐 나갔기 때문이다. 학습용 점수가 더 높은 이유는 학습용 사진은 모델이 여러 번 본 문제이고, 테스트용은 처음 보는 사진이기 때문이다. 두 선의 차이가 작으므로 모델이 외우기만 한 것이 아니라 처음 보는 사진에도 잘 맞히는 **일반화**가 되었다고 볼 수 있다.

---
### 논술 3. epoch을 3에서 50으로 늘리면 항상 더 좋아질까? 이유를 쓰시오.
**답**: 항상 좋아지지는 않는다. epoch을 늘리면 학습용 loss는 계속 줄어들지만, 어느 순간부터 테스트용 loss는 줄지 않거나 오히려 **올라갈 수 있다**. 이는 모델이 학습용 사진의 특징을 넘어 **세세한 부분까지 외워버린 과대적합** 때문이다. 그래서 학습 곡선에서 테스트용(val_) 선이 더 이상 좋아지지 않는 지점에서 멈추는 것이 좋으며, 학습 시간도 더 오래 걸린다.

---
### 논술 4. 노드 수(128개 vs 16개)에 따른 결과를 비교하고 해석하시오. (13차시)
**결과**: 노드 128개 모델의 테스트 정확도는 약 **0.97**, 노드 16개 모델은 약 **0.94**로 128개가 약 3%p 높았다.

**해석**: 노드 수가 많을수록 파라미터(가중치)가 많아져(**101,770개 vs 12,730개**) 손글씨의 더 다양한 특징을 표현할 수 있으므로 정확도가 높다. 하지만 파라미터가 약 8배 늘어난 것에 비해 정확도는 3%p 정도만 올랐고, 계산 시간과 메모리는 더 든다.

**결론**: 노드가 많을수록 대체로 정확도가 높지만, 늘어나는 계산량에 비해 효과가 점점 작아지므로 데이터에 맞는 적당한 크기를 고르는 것이 중요하다.

---
### 논술 5. AI가 틀린 사진들을 보고 그 이유를 분석하시오.
**결과**: AI는 테스트용 10000장 중 약 **1000장**(약 10%)을 틀렸다. 틀린 사진은 주로 **Shirt(셔츠)를 T-shirt·Pullover·Coat로**, 또는 그 반대로 헷갈린 경우가 많았다.

**해석**: 셔츠, 티셔츠, 풀오버, 코트는 모두 몸통과 소매가 있는 **비슷한 모양의 윗옷**이라 28×28 흑백 사진에서는 단추·옷깃 같은 작은 차이가 잘 보이지 않는다. 사람이 봐도 헷갈리는 사진이 많았다. 반면 바지(Trouser)나 가방(Bag)처럼 모양이 뚜렷하게 다른 옷은 거의 틀리지 않았다.

**결론**: 모양이 비슷한 종류일수록 구별이 어렵다. 더 큰 해상도나 컬러 사진, 더 많은 학습 데이터를 쓰면 이런 실수를 줄일 수 있을 것이다.

---
### 논술 6. 옷 사진만 학습한 AI에게 고양이 사진을 보여주면 어떻게 되는지, 이유를 쓰시오.
**결과**: AI는 고양이 사진을 **옷 10종류 중 하나**(예: Bag 이나 Shirt)로 답했고, 확신이 꽤 높게 나오기도 했다.

**해석**: 이 AI는 옷 10종류만 배웠고 출력층도 10개의 옷 중 하나를 고르도록 되어 있어서, "고양이"나 "모르겠다"라고 답할 방법이 없다. softmax는 항상 10개 확률의 합을 1로 만들기 때문에, 처음 보는 사진이어도 그중 가장 비슷한 옷을 고른다.

**결론**: AI는 **배운 것만 안다**. 학습 데이터의 범위를 벗어난 입력에는 틀린 답을 자신 있게 낼 수 있으므로, AI를 쓸 때는 어떤 데이터로 학습했는지 알고 그 범위 안에서 써야 한다.

---
### 논술 7. Conv2D 필터 수(16 / 32 / 64) 또는 층 수를 바꿨을 때 예상되는 변화를 쓰시오.
**답**: 필터 수를 늘리면 찾을 수 있는 모양의 종류가 많아져 정확도가 오를 수 있지만, 출력 채널이 늘어 Flatten 크기와 파라미터가 커진다(16개 27,210 → 32개 54,410 → 64개 108,810). Conv2D + MaxPooling2D를 한 층 더 쌓으면 이미지가 더 작아져(5×5×64 = 1600) 파라미터는 오히려 줄고(34,826개), 앞 층이 찾은 선·모서리를 조합해 더 큰 모양(소매, 옷깃 등)을 찾을 수 있어 정확도가 오르는 경우가 많다. 단, 실제 결과는 실행해서 테스트용 정확도로 확인해야 한다.

---
## 시험 직전 체크리스트
- [ ] `X_train`, `X_test` **둘 다** `/ 255` 했나?
- [ ] CNN이면 `reshape(-1, 28, 28, 1)` 했나? (`Input` 모양과 똑같아야 함)
- [ ] 출력층 노드 수 = **분류할 종류 수** 인가?
- [ ] `compile` → `fit` 순서, `epochs=` 숫자가 문제와 같은가?
- [ ] 곡선/비교에서 테스트용은 **`val_`** 로 시작하는 이름인가?
- [ ] `cv2.resize(img, (너비, 높이))` 순서가 맞나?
- [ ] 논술형은 **결과(숫자) → 해석(왜) → 결론** 순서로, 숫자는 내 실행 결과로 바꿨나?